# Forecasting rider fare pressure
### Victor H. Aguiar · Local NYC Uber demonstration

**Question:** Does a frozen time-series foundation model improve out-of-time rider-fare forecasts over strong seasonal and lagged-feature baselines, and does synthetic economic adaptation help or hurt?

**Target:** Hourly median realized passenger fare within fixed origin–destination zones, before taxes, fees, tolls and tips. Horizon h predicts the pickup-hour bin ending at origin + h, not an instantaneous price. This is **not** an upfront quote, identified surge premium, or driver-pay forecast. Uber One membership is not observed.

This notebook is a read-only walkthrough of saved results. Running all cells does **not** download data, fit models or send data anywhere.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

candidates = [Path.cwd(), Path.cwd() / 'research/uber_rider_fares']
ROOT = next((p for p in candidates if (p / 'protocol.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the demo folder or project root.')
OUT = ROOT / 'outputs'
required = ['metrics.json', 'run_manifest.json', 'test_predictions.parquet', 'report_metrics.json']
missing = [name for name in required if not (OUT / name).exists()]
if missing:
    raise FileNotFoundError(f'Run the benchmark/report deliberately first. Missing cached outputs: {missing}')
metrics = json.loads((OUT / 'metrics.json').read_text())
manifest = json.loads((OUT / 'run_manifest.json').read_text())
report = json.loads((OUT / 'report_metrics.json').read_text())
protocol = manifest['protocol']
display(Markdown(f"Test window ends {protocol['test_end']} UTC; horizons {protocol['horizons']} hours; history embargo {protocol['embargo_hours']} hours."))

## 1. Compare the models honestly
The final test table includes every model on matched observed cases. The named Chronos-versus-baseline comparison is chosen with validation, not test outcomes. Selected learned baselines are refitted on training plus available validation outcomes before test. A foundation-model win is not assumed.

In [ ]:
table = pd.DataFrame(metrics['test'])
table = table[table['horizon'].astype(str).eq('all')].sort_values('mae')
display(table[['model', 'n', 'mae', 'rmse', 'high_fare_mae', 'coverage_80', 'width_80']].round(3))
c = metrics['comparison']
lo, hi = c['paired_daily_block_bootstrap_95pct']
display(Markdown(f"**Validation-selected comparison:** {c['chronos']} versus {c['baseline']}. Chronos relative MAE improvement: {c['chronos_relative_mae_improvement_pct']:.1f}% (descriptive paired day-block interval {lo:.1f}% to {hi:.1f}%). A negative value means Chronos is worse."))

In [ ]:
display(Image(filename=str(OUT / 'figures/forecast_errors.png')))

## 2. Inspect a fixed example
The route is the first in the January-only ranking; the window is the first test week; the horizon is three hours. This plot was not selected because it makes Chronos look good. Differences can reflect changing service-tier or rider mix as well as pricing.

In [ ]:
display(Image(filename=str(OUT / 'figures/first_route_first_week.png')))
predictions = pd.read_parquet(OUT / 'test_predictions.parquet')
sample_route = report['example']['series_id']
sample = predictions[predictions.series_id.astype(str).eq(sample_route) & predictions.horizon.eq(3)]
sample = sample.sort_values(['origin', 'model']).head(16)
display(sample[['origin', 'target_timestamp', 'model', 'actual', 'prediction', 'q10', 'q90']].round(2))

## 3. What makes the test credible?

- January-only route selection; chronological train/validation/test windows.
- Common targets and a three-hour post-bin information embargo. Future realized fares, counts and durations are not model inputs.
- Missing low-volume outcomes remain missing. Local calendar features respect NYC daylight saving; storage uses a regular UTC grid.
- The cached original model weights are hash-identical to the October 2025 weights, before the 2026 data. The new synthetic LoRA adapter is kept separate and uses no real validation/test gradient updates.
- Baseline intervals use validation residuals; Chronos intervals use native quantiles. Coverage is measured rather than assumed.
- Public TLC releases are delayed: this is a retrospective pseudo-online comparison, not a live public feed.
- Day-block intervals are descriptive for this one-city window. Adjacent days can remain dependent.

In [ ]:
display(pd.Series({
    'Chronos snapshot': protocol['chronos_revision'],
    'Original weights published': protocol['original_weights_published'],
    'Weights SHA-256': manifest['model_hash'],
    'Device': 'Local CPU, float32',
    'Context hours': protocol['context_hours'],
    'Cross-learning across origins': protocol['chronos_cross_learning'],
    'Run finished (UTC)': manifest.get('finished_utc', 'Not recorded'),
}))

## 4. Economic simulation and adaptation

The simulator is motivated by the price-and-quoted-wait choice problem in [Buchholz et al. (Econometrica, 2025)](https://doi.org/10.3982/ECTA18838). It is a two-route hourly capacity/queue pilot, not a replication, estimated demand model or fleet-conserving spatial equilibrium.

Positive heterogeneous price/time sensitivities, outside options, FIFO queues, cancellations, shocks and pricing rules are randomized across synthetic episodes. Driver capacity is exogenous across policies; rider price changes do not automatically increase driver pay or supply. Only coarse NYC training-period fare/count scales calibrate the simulator.

The bounded local adaptation uses 60 LoRA steps with prespecified checkpoints 30 and 60. Step zero remains an option. The best adapted candidate is reported diagnostically even if validation recommends keeping the original.

In [ ]:
training_path = OUT / 'synthetic/fine_tuning_manifest.json'
if training_path.exists():
    training = json.loads(training_path.read_text())
    display(pd.Series({key: training.get(key) for key in ['status', 'optimizer_steps_completed', 'trainable_parameters', 'optimization_seconds', 'original_base_weights_unchanged']}))
    display(manifest.get('synthetic_adaptation', {}).get('validation_recommendation', 'See saved adaptation selection'))
else:
    print('No synthetic adaptation in this run.')

### Policy illustrations are not empirical treatment effects
Compare rules only within a simulated world. Potential-arrival, capacity and choice-uniform paths are fixed within that comparison. Worlds differ in multiple parameters. The anticipatory rule uses calendar and lagged potential arrivals, not Chronos forecasts. Its quoted ETA is a proxy rather than realized pickup waiting time. Revenue is gross rider fares before costs, not profit.

In [ ]:
stress_path = OUT / 'synthetic/stress_policy_comparison.json'
if stress_path.exists():
    stress = json.loads(stress_path.read_text())
    rows = [{**row, 'world': world['world']} for world in stress['worlds'] for row in world['rows']]
    policy_table = pd.DataFrame(rows)
    display(policy_table[['world', 'policy', 'requested_over_potential', 'completed_over_potential', 'mean_quoted_eta_minutes_weighted_by_potential', 'gross_rider_fare_revenue_usd_before_costs', 'ending_queue']].round(3))

## 5. From forecasting to a pricing decision

My science–engineering starting point would be a shared decision and data contract, a latency budget, reliable fallback behavior, and joint ownership through validation and rollout. A shadow forecast can establish reliability before a controlled product evaluation.

Economics connects forecast uncertainty to rider decisions: book, wait, switch product, switch platform or abandon. Subscription benefits could change that response, but this dataset cannot identify Uber One's effect. Inside Uber, offer sets, rejected quotes, benefits, membership histories and marketplace outcomes would let us evaluate conversion, retention, contribution margin and reliability jointly.

**What this does not claim:** optimal surge pricing, causal price elasticity, improvement over Uber's internal models, or evidence that a subscription policy works.

## Optional deliberate reproduction
These are commands to copy into a terminal, not cells that execute automatically. The data preparation step downloads several large official Parquet files if they are not already cached.

```bash
# Run from this cloned repository's root; see docs/reproduce.md.
.venv/bin/python prepare_data.py
.venv/bin/python benchmark.py --synthetic-checkpoints outputs/synthetic/checkpoints
.venv/bin/python report.py
```

[NYC TLC source](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) · [Field dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_hvfhs.pdf) · [Chronos-2](https://huggingface.co/amazon/chronos-2)